# 01. Tree of Thoughts

Tree of Thoughts (ToT) 将 Chain-of-Thought 从单条路径扩展为搜索树：生成多个候选「思维」→ 评估 → 剪枝 → 继续探索最有希望的路径。

**核心操作**：
- **Generate**: 从当前状态生成 N 个候选下一步
- **Evaluate**: 对每个候选打分（启发式或模型评分）
- **Select**: 选择 top-K 进入下一轮

> **检查点**：能画出 BFS 和 Beam Search 在 ToT 中的区别图。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
from dataclasses import dataclass

@dataclass
class State:
    numbers: tuple; expr: str = ""; depth: int = 0
    def score(self, target=24) -> float:
        return -min(abs(n-target) for n in self.numbers)

def children(state: State) -> list[State]:
    results = []
    nums = state.numbers
    for i in range(len(nums)):
        for j in range(len(nums)):
            if i == j: continue
            rest = [nums[k] for k in range(len(nums)) if k not in (i,j)]
            a, b = nums[i], nums[j]
            for val, op in [(a+b,f"({a}+{b})"),(a-b,f"({a}-{b})"),(a*b,f"({a}*{b})")]:
                results.append(State(tuple(rest+[val]), op, state.depth+1))
            if b != 0:
                results.append(State(tuple(rest+[a/b]), f"({a}/{b})", state.depth+1))
    return results

def is_goal(s: State, t=24) -> bool:
    return len(s.numbers)==1 and abs(s.numbers[0]-t)<0.001

# Beam Search
initial = State((4,7,8,8)); beam = [initial]; solved = None
for _ in range(6):
    candidates = []
    for s in beam:
        if is_goal(s): solved = s; break
        candidates.extend(children(s))
    if solved: break
    candidates.sort(key=lambda s: s.score(), reverse=True)
    beam = candidates[:3]

if solved:
    path = []; cur = solved
    while cur:
        path.append(cur.expr or f"{cur.numbers}")
        cur = getattr(cur,'parent',None)
    print(f"Solved in {len(path)-1} steps:")
    [print(f"  {s}") for s in reversed(path)]
else:
    print("No solution found")

No solution found


## Beam Search 参数的影响

| Beam Width | 搜索空间 | 找到解的概率 | 计算开销 |
|------------|----------|-------------|----------|
| 1 | 贪心（只看最优） | 低 | 最低 |
| 3 | 适中 | 中 | 中等 |
| 10 | 广 | 高 | 高 |

**启发式函数的设计至关重要**：差的评分函数即使 beam width 很大也会走偏。

> **检查点**：能解释为什么「beam width = 1」等价于贪心搜索，什么时候它会失败。

## 真实 API 实验

LLM 作为「思维生成器」跑 Tree of Thoughts（`tot_solver_demo.py --strategy llm`）。


In [3]:
# LLM 驱动的 ToT 求解 24 点
SCRIPT = ROOT / 'agents' / 'planning' / 'scripts' / 'tot_solver_demo.py'
import subprocess, sys
result = subprocess.run([sys.executable, str(SCRIPT), '--strategy', 'llm', '--numbers', '3', '3', '8', '8', '--use-api'],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)
    print("（LLM 未在限定深度内解出也属正常 — 观察它生成的候选 move 是否合法）")


Problem: make 24 from [3, 3, 8, 8]
Strategy: llm (max_depth=6)

Solution path (0 steps):
  No solution found

（LLM 未在限定深度内解出也属正常 — 观察它生成的候选 move 是否合法）


## 练习

1. 修改 `score()` 函数：改为鼓励「数字数量少」+「接近目标」的加权评分。对比效果。
2. 实现 DFS 搜索，并与 Beam Search 在相同的 24 点题目上对比（步数、解质量）。
3. 思考：ToT 的 `score()` 如果由 LLM 来评估会怎样？什么情况下 LLM 评分不可靠？

> **检查点**：能在白板上画出 ToT 的搜索树，标注 BFS/DFS/Beam 三种策略的探索顺序。